[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C21_Frontier_Pretraining_Course/00_setup/00_environment_check.ipynb)

# 00 · 课程总览与环境检查

这个 notebook 做两件事：① **确认环境**（只需 numpy）；② 用几个**可运行的小演示**预览本课五个模块各自要解决的问题，建立全局直觉。

> 全课纯 numpy / CPU。不训真模型，而是用玩具规模把每个预训练机制的**算法骨架与正确性**夯实。每个机制都与朴素参考**对拍**、用 `assert` 兜底。

## 1 · 环境检查

只需要 numpy。下面确认它能用，并固定随机种子以便结果可复现。

In [ ]:
import numpy as np
import sys, hashlib, math
from collections import Counter, defaultdict

print('Python', sys.version.split()[0])
print('numpy ', np.__version__)
rng = np.random.default_rng(0)
assert np.__version__ >= '1.20'
x = rng.standard_normal((3, 3))
assert x.shape == (3, 3)
print('\n✅ 环境就绪：numpy 可用，全课 CPU 秒级运行')

## 2 · 预览模块 01：去重为什么需要「近似」

网页语料里全是**近重复**（转载、改一个词）。逐字 hash 抓不到它们。本课模块 01 用 **MinHash** 把「比集合」变成「比短签名」。

先直观看一下：两篇只差几个词的文档，Jaccard 相似度依然很高——这正是逐字 hash 会漏、而我们需要近似方法的原因。

In [ ]:
def shingles(text, k=2):
    '''把文本按词切成 k-gram 集合(shingle 集合)，用于算 Jaccard。'''
    toks = text.split()
    return set(tuple(toks[i:i+k]) for i in range(len(toks) - k + 1))

def jaccard(a, b):
    return len(a & b) / len(a | b) if (a | b) else 0.0

doc1 = 'the quick brown fox jumps over the lazy dog every morning'
doc2 = 'the quick brown fox leaps over the lazy dog every morning'   # 改了一个词
doc3 = 'a completely different sentence about machine learning models'

s1, s2, s3 = shingles(doc1), shingles(doc2), shingles(doc3)
print(f'近重复 doc1 vs doc2 的 Jaccard = {jaccard(s1, s2):.3f}  (高 → 是近重复)')
print(f'无关   doc1 vs doc3 的 Jaccard = {jaccard(s1, s3):.3f}  (低 → 不相关)')
assert jaccard(s1, s2) > 0.6, '改一个词仍应高度相似'
assert jaccard(s1, s3) < 0.1, '无关文档应低相似'
print('\n✅ 逐字 hash 会把 doc1/doc2 当成两篇不同文档(漏掉近重复)；模块 01 用 MinHash/LSH 抓它们')

## 3 · 预览模块 02：tokenizer 的压缩率

tokenizer 把文本切成 token。同样一段文本，切出的 token 越少（fertility 越低 / 压缩率越高），序列越短、训练推理越省。

这里用「按字符切」对比「按词切」，直观感受**压缩率**这个模块 02 的核心指标。

In [ ]:
text = 'the quick brown fox jumps over the lazy dog'
n_bytes = len(text.encode('utf-8'))

char_tokens = list(text)                 # 最细：每个字符一个 token
word_tokens = text.split()               # 最粗：每个词一个 token

def compression_ratio(n_bytes, n_tokens):
    return n_bytes / n_tokens             # 每个 token 平均顶多少字节，越大越好

print(f'原始 {n_bytes} 字节')
print(f'按字符: {len(char_tokens):2d} tokens -> 压缩率 {compression_ratio(n_bytes, len(char_tokens)):.2f} 字节/token')
print(f'按词:   {len(word_tokens):2d} tokens -> 压缩率 {compression_ratio(n_bytes, len(word_tokens)):.2f} 字节/token')
assert compression_ratio(n_bytes, len(word_tokens)) > compression_ratio(n_bytes, len(char_tokens))
print('\n✅ 词级压缩率更高(序列更短)，但词表会爆炸且有 OOV —— BPE 是两者的折中(模块 02)')

## 4 · 预览模块 03/04：尺度与稳定性

大模型训练的两个核心数值问题：① 让激活/梯度的**尺度**在变宽时别漂移（模块 03 μP）；② 让梯度别突然爆炸引发 **loss spike**（模块 04）。

下面演示「梯度裁剪」——模块 04 最基础的护栏：当梯度范数超阈值就等比缩小，方向不变、大小受控。

In [ ]:
def clip_grad(g, max_norm):
    '''全局梯度裁剪：范数超过 max_norm 就等比例缩放，否则不动。'''
    norm = np.linalg.norm(g)
    if norm > max_norm:
        g = g * (max_norm / norm)
    return g, norm

rng = np.random.default_rng(1)
normal_grad = rng.standard_normal(100) * 0.1     # 正常的小梯度
spike_grad  = rng.standard_normal(100) * 50.0    # 一个会引爆训练的尖峰梯度

for name, g in [('正常梯度', normal_grad), ('尖峰梯度', spike_grad)]:
    clipped, before = clip_grad(g, max_norm=1.0)
    print(f'{name}: 裁剪前范数 {before:7.2f} -> 裁剪后范数 {np.linalg.norm(clipped):.4f}')

_, before = clip_grad(spike_grad, 1.0)
clipped, _ = clip_grad(spike_grad, 1.0)
assert np.isclose(np.linalg.norm(clipped), 1.0), '尖峰梯度应被裁到范数=1'
# 裁剪不改方向，只改大小
assert np.allclose(clipped / np.linalg.norm(clipped), spike_grad / np.linalg.norm(spike_grad))
print('\n✅ 梯度裁剪把尖峰梯度的范数压到阈值、方向不变 —— 模块 04 最基础的 spike 护栏')

## 5 · 预览模块 05：数据配比是个优化问题

各 domain（网页/代码/书/多语）按什么比例采样？这是一个**优化问题**。模块 05 会讲 DoReMi 怎么自动找权重。

先看最简单的事实：配比就是一个**和为 1 的概率向量**，不同配比会让模型在各 domain 上的（加权）loss 不同。

In [ ]:
domains = ['web', 'code', 'books', 'multilingual']
# 假设各 domain 单独训练能达到的 loss(越小越好)与其数据丰富度
per_domain_loss = np.array([2.8, 2.2, 2.5, 3.1])

def mixture_loss(weights, losses):
    '''给定配比权重(和为1)，加权 loss。'''
    weights = np.asarray(weights, dtype=float)
    assert np.isclose(weights.sum(), 1.0), '配比权重必须和为 1'
    return float(weights @ losses)

uniform = np.ones(4) / 4
web_heavy = np.array([0.7, 0.1, 0.1, 0.1])
print(f'均匀配比     loss = {mixture_loss(uniform, per_domain_loss):.3f}')
print(f'偏重 web     loss = {mixture_loss(web_heavy, per_domain_loss):.3f}')
assert np.isclose(mixture_loss(uniform, per_domain_loss), per_domain_loss.mean())
print('\n✅ 配比 = 和为 1 的权重向量；模块 05 用 DoReMi 自动找「让最差 domain 也学好」的权重')

---
## ✏️ 练习：实现一个「数据墙」收益递减曲线

模块 05 的核心结论之一：数据有限时重复 epoch，**收益递减**。一个常用的简化模型是：训练 R 个 epoch 的「有效数据量」不是 `R × D`，而是带衰减的 `D × (1 - exp(-R/tau)) × tau`（每多重复一遍，新增有效数据越来越少）。

实现 `effective_data(D, R, tau)`，并验证它**随 R 单调增、但边际递减**。

In [ ]:
def effective_data(D, R, tau=4.0):
    '''重复 R 个 epoch 的有效数据量(简化模型，体现收益递减)。
       提示：D * tau * (1 - exp(-R/tau))。R 小时≈R*D(线性)，R 大时饱和到 D*tau。'''
    # TODO: 实现上面的公式
    raise NotImplementedError

In [ ]:
# —— 练习自测 ——
D = 100.0
vals = [effective_data(D, R) for R in [1, 2, 3, 4, 5]]   # 等间隔 R，才能比每步增量
print('R=1..5 的有效数据量:', [round(v, 1) for v in vals])
# 单调增
assert all(vals[i] < vals[i+1] for i in range(len(vals)-1)), '应随 R 单调增'
# 边际递减：等间隔 R 下，每一步的增量越来越小
deltas = [vals[i+1] - vals[i] for i in range(len(vals)-1)]
assert all(deltas[i] > deltas[i+1] for i in range(len(deltas)-1)), '增量应递减'
# 饱和：R 很大时趋向 D*tau
assert effective_data(D, 1000) < D * 4.0 + 1e-6
assert effective_data(D, 1000) > D * 4.0 * 0.99
print('✅ 练习通过：有效数据随重复递增但边际递减，最终饱和(这就是「数据墙」的数学形状)')

---
### 📖 参考答案

In [ ]:
# 练习 参考答案
def effective_data(D, R, tau=4.0):
    return D * tau * (1.0 - math.exp(-R / tau))

---
## 🧪 真实数据胶囊：Chinchilla 的 20:1 法则算一笔账

Chinchilla（Hoffmann 2022）的核心结论：给定算力，**训练 token 数 D ≈ 20 × 参数量 N** 时最优。
训练算力近似 `C ≈ 6 N D`（FLOPs）。用这个关系给几个真实规模的模型算「算力最优」该喂多少 token。

In [ ]:
def chinchilla_optimal_tokens(N, ratio=20):
    '''Chinchilla：算力最优的训练 token 数 ≈ 20 × 参数量。'''
    return ratio * N

def training_flops(N, D):
    '''标准近似：前向+反向 ≈ 6 FLOPs/参数/token。'''
    return 6 * N * D

for name, N in [('GPT-3 175B', 175e9), ('Chinchilla 70B', 70e9), ('Llama-2 7B', 7e9)]:
    D = chinchilla_optimal_tokens(N)
    C = training_flops(N, D)
    print(f'{name:18s} N={N/1e9:6.0f}B -> 最优 D≈{D/1e9:6.0f}B token, 训练 C≈{C:.1e} FLOPs')

# GPT-3 实际只训了 ~300B token(远少于 175*20=3500B) —— 这正是 Chinchilla 指出的「训练不足」
gpt3_optimal = chinchilla_optimal_tokens(175e9)
gpt3_actual = 300e9
assert gpt3_optimal > 10 * gpt3_actual, 'GPT-3 按 Chinchilla 应训多得多的 token'
print(f'\nGPT-3 实际训 ~300B，Chinchilla 最优 ~{gpt3_optimal/1e9:.0f}B —— 严重训练不足(模块 05 详述)')

**🧪 胶囊练习**：实现 `params_for_budget(C, ratio=20)`：给定算力预算 C（FLOPs），在 D=ratio·N 的约束下，反解出算力最优的参数量 N。

提示：`C = 6·N·D = 6·N·(ratio·N) = 6·ratio·N²`，所以 `N = sqrt(C / (6·ratio))`。

In [ ]:
def params_for_budget(C, ratio=20):
    # TODO: 由 C = 6*ratio*N^2 反解 N
    raise NotImplementedError

In [ ]:
# 自测
C = 6 * 70e9 * (20 * 70e9)        # 用 Chinchilla-70B 的算力反推
N = params_for_budget(C)
print(f'算力 C={C:.1e} FLOPs -> 最优参数量 N≈{N/1e9:.1f}B')
assert abs(N - 70e9) / 70e9 < 0.01, '应反解回 ~70B'
print('✅ 胶囊练习通过：给定算力能反解出最优模型大小')

In [ ]:
# 📖 胶囊参考答案
def params_for_budget(C, ratio=20):
    return math.sqrt(C / (6 * ratio))

### 小结与下一步
- 预训练 = 一条从 CommonCrawl 到 checkpoint 的长流水线；本课拆解其中**数据/超参/稳定性**五个决定成败的非架构环节。
- 方法论：纯 numpy 玩具规模复现**算法与正确性**，每步与朴素参考**对拍**、`assert` 兜底；放大到集群是 C36/C39 的事。
- 你已预览：去重需近似(MinHash)、tokenizer 看压缩率、训练要控尺度(μP)与防 spike(裁剪)、配比是优化问题、数据有墙(Chinchilla/重复递减)。

下一站：**模块 01 · 数据清洗与去重** —— 从原始网页到干净不重复的语料，质量过滤 + MinHash/LSH 去重 + n-gram 去污染，全程从零实现。